# CBRAIN OASIS Interpretation (Step 4)

Goal: produce a leakage-aware interpretation layer from frozen Step 3 artifacts without any new model fitting.


In [ ]:
# 1) Load frozen modeling artifacts
from pathlib import Path
import json
import numpy as np
import pandas as pd
import textwrap

ROOT = Path.cwd()
results_dir = ROOT / 'results' / 'modeling'
docs_dir = ROOT / 'docs'

coeff_df = pd.read_csv(results_dir / 'baseline_logistic_coefficients.csv')
rf_imp_df = pd.read_csv(results_dir / 'step3b_random_forest_feature_importance.csv')
baseline_metrics = json.loads((results_dir / 'baseline_metrics.json').read_text())
rf_metrics = json.loads((results_dir / 'step3b_random_forest_metrics.json').read_text())

print('Loaded artifacts from', results_dir)
coeff_df


In [ ]:
# 2) Build cross-model interpretation table
feature_map = {
    'Age_z': 'Age',
    'Gender_code': 'Gender',
    'EDUC_z': 'EDUC',
    'SES_imputed_z': 'SES',
    'SES_missing': 'SES_missing_indicator',
    'eTIV_z': 'eTIV',
    'nWBV_z': 'nWBV',
    'ASF_z': 'ASF',
}

coef = coeff_df.loc[coeff_df['feature'] != 'intercept', ['feature', 'coefficient']].copy()
coef['predictor'] = coef['feature'].map(feature_map).fillna(coef['feature'])
coef['abs_coefficient'] = coef['coefficient'].abs()
coef['logistic_rank'] = coef['abs_coefficient'].rank(method='dense', ascending=False).astype(int)
coef['direction'] = np.where(coef['coefficient'] >= 0, 'positive_association', 'negative_association')

rf = rf_imp_df.copy()
rf['predictor'] = rf['feature'].map(feature_map).fillna(rf['feature'])
rf['rf_rank'] = rf['importance'].rank(method='dense', ascending=False).astype(int)

interp_df = coef.merge(
    rf[['predictor', 'importance', 'rf_rank']],
    on='predictor',
    how='left'
).sort_values(['logistic_rank', 'rf_rank', 'predictor']).reset_index(drop=True)

interp_df


In [ ]:
# 3) Add biological-context notes and materialize Step 4 outputs
context_notes = {
    'Age': 'Age captures broad neurodegenerative risk trends but can proxy survival and selection effects.',
    'Gender': 'Sex/gender differences may reflect both biology and cohort sampling artifacts.',
    'EDUC': 'Education can represent cognitive reserve and social determinants, not direct pathology.',
    'SES': 'SES links to access and life-course exposure; interpretation is confounded by social factors.',
    'SES_missing_indicator': 'Missingness itself may encode collection bias; treat as data-quality signal.',
    'eTIV': 'eTIV is structural context and not a direct disease marker in isolation.',
    'nWBV': 'nWBV is a global atrophy proxy and biologically plausible in cognitive decline patterns.',
    'ASF': 'ASF is structural scaling and should be interpreted cautiously due to collinearity.',
}

interp_df['biological_context_note'] = interp_df['predictor'].map(context_notes).fillna('Descriptive predictor only.')

step4_csv = results_dir / 'step4_feature_interpretation.csv'
step4_json = results_dir / 'step4_interpretation_summary.json'
interp_df.to_csv(step4_csv, index=False)

summary_payload = {
    'stage': 'step4_interpretation',
    'date': '2026-03-23',
    'primary_model': 'baseline_logistic_sklearn',
    'comparison_model': 'random_forest_sklearn',
    'validation_metrics': {
        'baseline_logistic': baseline_metrics['validation_metrics'],
        'random_forest': rf_metrics['validation_metrics'],
    },
    'top_logistic_abs_coef_predictors': interp_df.sort_values('abs_coefficient', ascending=False)['predictor'].head(3).tolist(),
    'top_random_forest_predictors': interp_df.sort_values('importance', ascending=False)['predictor'].head(3).tolist(),
    'causal_guardrail': 'All interpretations are associative only; causal claims are not valid from this design.',
}
step4_json.write_text(json.dumps(summary_payload, indent=2))

print('Wrote', step4_csv)
print('Wrote', step4_json)
interp_df


In [ ]:
# 4) Write Step 4 interpretation report
core_cols = ['predictor', 'coefficient', 'abs_coefficient', 'logistic_rank', 'importance', 'rf_rank', 'direction']
report_table = interp_df[core_cols].copy()
report_md = report_table.to_markdown(index=False, floatfmt='.4f')

doc_text = f"""# Step 4: Model Interpretation and Scientific Framing

Date: March 23, 2026 (IST)

## Scope

This stage translates frozen Step 3 outputs into a conservative interpretation layer.
No additional training or resplitting is performed.

## Quantitative Feature Signals

Cross-model view (logistic coefficients + random-forest importance):

{report_md}

## Scientific Reading (Non-Causal)

- nWBV and Age remain high-signal features across models, consistent with expected aging/atrophy associations.
- EDUC and SES should be treated as cognitive-reserve and socioeconomic context variables, not mechanistic biomarkers.
- Gender and SES_missing_indicator can reflect cohort or data-collection effects as much as biology.
- eTIV and ASF are structural scaling features and should not be overinterpreted in isolation.

## Strict Guardrails

- Associations only: this is not causal inference.
- Validation size is small (n=33), so rank stability is limited.
- Single split only; no external cohort or nested CV yet.
- Interpretation reflects current feature engineering choices and may shift after stronger uncertainty analyses.

## Artifacts

- results/modeling/step4_feature_interpretation.csv
- results/modeling/step4_interpretation_summary.json
"""

step4_doc = docs_dir / 'interpretation_step4.md'
step4_doc.write_text(doc_text.strip() + "\n")
print('Wrote', step4_doc)


In [ ]:
# 5) Contract checks
required_cols = {
    'predictor',
    'coefficient',
    'abs_coefficient',
    'logistic_rank',
    'importance',
    'rf_rank',
    'direction',
    'biological_context_note',
}
assert required_cols.issubset(set(interp_df.columns)), 'Missing expected interpretation columns'
assert interp_df['predictor'].is_unique, 'Predictor rows must be unique'
assert (results_dir / 'step4_feature_interpretation.csv').exists()
assert (docs_dir / 'interpretation_step4.md').exists()
print('Step 4 checks passed.')
